In [ ]:
import time

import torch
from torch import nn

from pract11.Guia11Reformat.src.ClassifierMLP import ClassifierMLP
from pract11.Guia11Reformat.src.Config import Config
from pract11.Guia11Reformat.src.FashionMNISTData import FashionMNISTData
from pract11.Guia11Reformat.src.Optimizer import create_optimizer
from pract11.Guia11Reformat.src.Trainer import Trainer
from pract11.Guia11Reformat.src.Visualizer import Visualizer


def runExperiment(config):
    # Configuración
    device = config.setup()
    # Datos
    data = FashionMNISTData(
        batch_size=config.BATCH_SIZE,
        seed=config.SEED, color=config.COLOR, show_dataset_info=config.show_dataset_info,
    )

    if config.show_sample:
        data.inspect_example()

    # Modelo
    torch.manual_seed(config.SEED)

    model = ClassifierMLP(
        p_dropout=config.DROPOUT_P
    ).to(device)

    # Comprobar dimensiones
    if config.show_dimentions:
        images, labels = next(iter(data.train_loader))

        with torch.no_grad():
            classifications = model(images.to(device))

        print(f"images.shape: {images.shape}")
        print(f"labels.shape: {labels.shape}")

        n_params = sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )
        print(f"Parámetros entrenables: {n_params:,}")

        print("\nDimensiones:")
        print("Entrada:", images.shape)
        print("Logits:", classifications.shape)

    # Función de pérdida y optimizador
    loss_fn = nn.CrossEntropyLoss()

    optimizer = create_optimizer(
        model,
        config.OPTIMIZER_NAME
    )

    print("\nOptimizador:", config.OPTIMIZER_NAME)
    print("Dropout:", config.DROPOUT_P)

    # Untrained
    if config.show_untrained:
        Visualizer.showPredictions(model, device, data)

    # Entrenamiento
    trainer = Trainer(
        model,
        optimizer,
        loss_fn,
        device,
        config
    )
    start = time.perf_counter()
    trainer.fit(
        data.train_loader,
        data.valid_loader,
        epochs=config.EPOCHS
    )

    print(f"Training time: {time.perf_counter() - start:.2f} seconds")
    # Gráfico de pérdidas
    if config.show_loss: Visualizer.plot_history(trainer.history)

    # Evaluación final
    _, _, confusion_matrix = trainer.test(data.test_loader)

    #Trained
    if config.show_trained:
        Visualizer.showPredictions(model, device, data)
    return trainer, data, confusion_matrix


# ============================================================
# EJECUCIÓN
# ============================================================
# showSample=False, showDimentions=False, showUntrained=False, showLoss=False, showTrained=True, showDimentions2=False, showEpochData=True
# runExperiment(Config(dropout_p=0.2, epochs=2)

expies = {
    "SGD": Config(dropout_p=0.2, epochs=20, optimizer_name="SGD", show_sample=False, show_dimentions=False,
                  show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                  show_epoch_data=False, show_init=True, show_dataset_info=False),
    "Adam": Config(dropout_p=0.2, epochs=20, optimizer_name="Adam", show_sample=False, show_dimentions=False,
                   show_dimentions2=False, show_untrained=False, show_loss=False, show_trained=False,
                   show_epoch_data=False, show_init=False, show_dataset_info=False)
}

results = {}
for title, exp in expies.items():
    trainer, data, confusion_matrix = runExperiment(exp)
    results[title] = {"trainer": trainer, "data": data, "confusion_matrix": confusion_matrix}
Visualizer.showExpieResults(results)



========== CONFIGURATION ==========
Seed:             42
Batch size:       100
Epochs:           20
Dropout:          0.2
Optimizer:        SGD
Adam_LR:        0.001
SGD_LR:        0.01

========== ENVIRONMENT ==========
PyTorch: 2.6.0+cu124
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


Optimizador: SGD
Dropout: 0.2

Iniciando entrenamiento... Opti -> SGD | Drop -> 0.2 | Batch size -> 100
